In [4]:
%%writefile extract_pq_jupyter.py
from odbAccess import openOdb
from abaqusConstants import INTEGRATION_POINT
import sys

def ExportRockStressesAllFrames(odb_path, output_file):
    try:
        odb = openOdb(path=odb_path)
    except Exception as e:
        print("Erro ao abrir o ODB: {}".format(e))
        return

    a = odb.rootAssembly
    set_name = 'FASEI_OPEN_WELL' 
    
    if set_name not in a.elementSets.keys():
        print("Erro: O Element Set '{}' nao foi encontrado.".format(set_name))
        odb.close()
        return
        
    target_set = a.elementSets[set_name]

    with open(output_file, 'w') as f:
        f.write("Time_s,Element_Label,p_PRESS_MPa,q_MISES_MPa\n")
        
        for step in odb.steps.values():
            for frame in step.frames:
                time = frame.frameValue
                
                # Extrai tensoes nos pontos de integracao
                press_field = frame.fieldOutputs['PRESSONLY'].getSubset(region=target_set, position=INTEGRATION_POINT)
                mises_field = frame.fieldOutputs['S'].getSubset(region=target_set, position=INTEGRATION_POINT)
                
                for press_val, s_val in zip(press_field.values, mises_field.values):
                    e_label = press_val.elementLabel
                    p_val = press_val.data
                    q_val = s_val.mises
                    
                    # Usa .format() para garantir compatibilidade com Abaqus/Python 2.7 ou 3
                    f.write("{},{},{},{}\n".format(time, e_label, p_val, q_val))
                    
    odb.close()
    print(">>> Sucesso! Curvas p-q exportadas para: " + output_file)

# Ponto de entrada quando executado pelo terminal do Abaqus
if __name__ == "__main__":
    # Substitua pelo caminho real do seu modelo no notebook
    odb_file = "WellClosureJob.odb" 
    csv_file = "dados_pq_sal.csv"
    ExportRockStressesAllFrames(odb_file, csv_file)

Overwriting extract_pq_jupyter.py


In [5]:
!abaqus python extract_pq_jupyter.py

**********************************************************************
** Visual Studio 2019 Developer Command Prompt v16.11.51
** Copyright (c) 2021 Microsoft Corporation
**********************************************************************
[vcvarsall.bat] Environment initialized for: 'x64'
Erro ao abrir o ODB: Cannot open file c:/Users/hidalgo/Documents/GitHub/Abaqus_WELL/WellClosureJob.odb. *** ERROR: No such file: c:/Users/hidalgo/Documents/GitHub/Abaqus_WELL/WellClosureJob.odb.


In [7]:
import pandas as pd

# Carrega o CSV gerado pelo Abaqus
df = pd.read_csv("dados_pq_sal.csv")

# Exibe as primeiras linhas para verificar os dados extraídos da rocha
# display(df.head())
display(df.tail())

df_creep = df[df['Time_s'] > 0.0]
display(df_creep.head())

,Time_s,Element_Label,p_PRESS_MPa,q_MISES_MPa
227455,1.892160e+09,7844,31406982.0,11083378.0
227456,1.892160e+09,7875,31383714.0,11065330.0
227457,1.892160e+09,7875,31383714.0,11065329.0
227458,1.892160e+09,7875,31383714.0,11063022.0
227459,1.892160e+09,7875,31383714.0,11063022.0


,Time_s,Element_Label,p_PRESS_MPa,q_MISES_MPa
1020,1.0,31,45813756.0,16.970562
1021,1.0,31,45813756.0,16.970562
1022,1.0,31,45813756.0,16.970562
1023,1.0,31,45813756.0,16.970562
1024,1.0,62,45831776.0,16.248077


In [6]:
%matplotlib qt

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Carregar os dados exportados
df = pd.read_csv("dados_pq_sal.csv")

# 2. Conversão de Unidades (Pascal para MPa)
# Se os seus dados no Abaqus já estiverem em MPa, comente as duas linhas abaixo.
df['p_PRESS_MPa'] = df['p_PRESS_MPa'] / 1e6
df['q_MISES_MPa'] = df['q_MISES_MPa'] / 1e6

# 3. Isolar um elemento crítico (ex: o primeiro elemento da lista, que costuma ser o da parede)
# Você pode alterar 'elemento_critico' manualmente para um número específico, ex: elemento_critico = 31
elemento_critico = df['Element_Label'].unique()[0]
# df_elem = df[df['Element_Label'] == elemento_critico].sort_values(by='Time_s')
# df_elem = df_elem.iloc[1:]


# Garante que os dados estão isolados (sem o .iloc[1:] que estava cortando apenas 1 linha)
df_elem = df[df['Element_Label'] == elemento_critico].copy()
# --- NOVO: Isolando a fase de Creep ---
# Encontra o índice onde o q atinge o pico máximo (início da relaxação)
indice_pico_q = df_elem['q_MISES_MPa'].idxmax()
# Filtra o dataframe para manter apenas os dados do pico em diante
df_elem = df_elem.loc[indice_pico_q:]
# --------------------------------------

p = df_elem['p_PRESS_MPa'].values
q = df_elem['q_MISES_MPa'].values

# 4. Configuração do Gráfico
plt.figure(figsize=(10, 6))

# Plotar a Trajetória de Tensões (Stress Path)
plt.plot(p, q, marker='o', linestyle='-', color='b', markersize=4, label=f'Trajetória - Elemento {elemento_critico}')

# Destacar o ponto inicial e final da simulação
plt.plot(p[0], q[0], marker='s', color='green', markersize=8, label='Beggining of Creep')
plt.plot(p[-1], q[-1], marker='X', color='red', markersize=8, label='End of Simulation')

# # Para plotar múltiplos elementos de uma vez:
# for elem in df['Element_Label'].unique()[:5]: # Plota apenas os 5 primeiros para não poluir
#     df_elem = df[df['Element_Label'] == elem].sort_values(by='Time_s')
#     plt.plot(df_elem['p_PRESS_MPa'], df_elem['q_MISES_MPa'], marker='.', linewidth=1, label=f'Elem {elem}')

# 5. Adicionar Linhas Geomecânicas (Parâmetros Didáticos/Hipotéticos)
# Ajuste as equações abaixo conforme as propriedades reais do seu sal
p_max = max(p) * 1.3 if len(p) > 0 and max(p) > 0 else 50
p_range = np.linspace(0, p_max, 100)

# Fronteira de Dilatância (Exemplo: q = 0.8 * p)
plt.plot(p_range, 0.8 * p_range, color='orange', linestyle='--', linewidth=2, label='Dilatance Boundary')

# Envelope de Ruptura (Exemplo: q = 5 + 1.2 * p)
plt.plot(p_range, 5.0 + 1.2 * p_range, color='darkred', linestyle='-', linewidth=2, label='Rupture Envelope')

# 6. Formatação Visual
# plt.title('Diagrama p-q: Fechamento do Espaço Anular', fontsize=14, fontweight='bold')
plt.title('p-q Diagram: Annular Space Closure', fontsize=14, fontweight='bold')
plt.xlabel('Mean Stress, p (MPa)', fontsize=12)
plt.ylabel('Deviatoric Stress, q (MPa)', fontsize=12)

# Define a origem dos eixos em zero para contexto físico
plt.xlim(left=0, right=p_max)
plt.ylim(bottom=0)

plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='upper left', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

In [9]:
%%writefile extract_strain_jupyter.py
from odbAccess import openOdb
from abaqusConstants import INTEGRATION_POINT
import sys

def ExportRockStrainsAllFrames(odb_path, output_file):
    try:
        odb = openOdb(path=odb_path)
    except Exception as e:
        print("Erro ao abrir o ODB: {}".format(e))
        return

    a = odb.rootAssembly
    set_name = 'FASEI_OPEN_WELL' 
    
    if set_name not in a.elementSets.keys():
        print("Erro: O Element Set '{}' nao foi encontrado.".format(set_name))
        odb.close()
        return
        
    target_set = a.elementSets[set_name]

    # Focando diretamente no passo de fluência
    target_step_name = 'Cas_9_875_Creep'

    with open(output_file, 'w') as f:
        # Cabeçalho do CSV
        f.write("Time_s,Element_Label,LE11,LE22,LE33,PEEQ\n")
        
        if target_step_name in odb.steps.keys():
            step = odb.steps[target_step_name]
            
            for frame in step.frames:
                time = frame.frameValue
                
                # Verifica se as variáveis existem no frame para evitar erros
                if 'LE' not in frame.fieldOutputs or 'PEEQ' not in frame.fieldOutputs:
                    continue
                
                le_field = frame.fieldOutputs['LE'].getSubset(region=target_set, position=INTEGRATION_POINT)
                peeq_field = frame.fieldOutputs['PEEQ'].getSubset(region=target_set, position=INTEGRATION_POINT)
                
                for le_val, peeq_val in zip(le_field.values, peeq_field.values):
                    e_label = le_val.elementLabel
                    
                    # No Abaqus, os índices do tensor de deformação são: 0->11(X), 1->22(Y), 2->33(Z)
                    le11 = le_val.data[0]
                    le22 = le_val.data[1]
                    le33 = le_val.data[2]
                    peeq = peeq_val.data
                    
                    f.write("{},{},{},{},{},{}\n".format(time, e_label, le11, le22, le33, peeq))
        else:
            print("Aviso: Step '{}' nao encontrado.".format(target_step_name))

    odb.close()
    print(">>> Sucesso! Dados de deformacao exportados para: " + output_file)

if __name__ == "__main__":
    odb_file = "WellClosureJob.odb" 
    csv_file = "dados_strain_sal.csv"
    ExportRockStrainsAllFrames(odb_file, csv_file)

Overwriting extract_strain_jupyter.py


In [10]:
%%writefile extract_strain_all_steps_jupyter.py
from odbAccess import openOdb
from abaqusConstants import INTEGRATION_POINT
import sys

def ExportRockStrainsAllFrames(odb_path, output_file):
    try:
        odb = openOdb(path=odb_path)
    except Exception as e:
        print("Erro ao abrir o ODB: {}".format(e))
        return

    a = odb.rootAssembly
    set_name = 'FASEI_OPEN_WELL' 
    
    if set_name not in a.elementSets.keys():
        print("Erro: O Element Set '{}' nao foi encontrado.".format(set_name))
        odb.close()
        return
        
    target_set = a.elementSets[set_name]

    with open(output_file, 'w') as f:
        # Adicionado 'Step_Name' no cabeçalho para controle e filtragem no Pandas
        f.write("Step_Name,Time_s,Element_Label,LE11,LE22,LE33,PEEQ\n")
        
        # Loop iterando por TODOS os steps existentes no modelo
        for step_name, step in odb.steps.items():
            for frame in step.frames:
                time = frame.frameValue
                
                # Checagem de segurança caso algum step inicial (ex: Geostatic) não tenha gerado PEEQ
                if 'LE' not in frame.fieldOutputs or 'PEEQ' not in frame.fieldOutputs:
                    continue
                
                le_field = frame.fieldOutputs['LE'].getSubset(region=target_set, position=INTEGRATION_POINT)
                peeq_field = frame.fieldOutputs['PEEQ'].getSubset(region=target_set, position=INTEGRATION_POINT)
                
                for le_val, peeq_val in zip(le_field.values, peeq_field.values):
                    e_label = le_val.elementLabel
                    
                    le11 = le_val.data[0]
                    le22 = le_val.data[1]
                    le33 = le_val.data[2]
                    peeq = peeq_val.data
                    
                    f.write("{},{},{},{},{},{},{}\n".format(step_name, time, e_label, le11, le22, le33, peeq))

    odb.close()
    print(">>> Sucesso! Dados de deformacao de todos os steps exportados para: " + output_file)

if __name__ == "__main__":
    odb_file = "WellClosureJob.odb" 
    csv_file = "dados_strain_all_sal.csv"
    ExportRockStrainsAllFrames(odb_file, csv_file)

Overwriting extract_strain_all_steps_jupyter.py


In [11]:
!abaqus python extract_strain_all_steps_jupyter.py
# !abaqus python extract_strain_jupyter.py

**********************************************************************
** Visual Studio 2019 Developer Command Prompt v16.11.51
** Copyright (c) 2021 Microsoft Corporation
**********************************************************************
[vcvarsall.bat] Environment initialized for: 'x64'
Erro ao abrir o ODB: Cannot open file c:/Users/hidalgo/Documents/GitHub/Abaqus_WELL/WellClosureJob.odb. *** ERROR: No such file: c:/Users/hidalgo/Documents/GitHub/Abaqus_WELL/WellClosureJob.odb.


In [12]:
!pip install PyQt5

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\hidalgo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [13]:
%matplotlib qt

import pandas as pd
import matplotlib.pyplot as plt

# 1. Carregar os dados exportados
df_strain = pd.read_csv("dados_strain_sal.csv")

# 2. Calcular a Deformação Volumétrica e converter para Porcentagem (%)
# Volumétrica = LE11 + LE22 + LE33
df_strain['Volumetric_Strain_pct'] = (df_strain['LE11'] + df_strain['LE22'] + df_strain['LE33']) * 100
df_strain['PEEQ_pct'] = df_strain['PEEQ'] * 100

# 3. Isolar o elemento crítico
elemento_critico = df_strain['Element_Label'].unique()[0]
df_elem = df_strain[df_strain['Element_Label'] == elemento_critico].sort_values(by='Time_s')

# 4. Configuração do Gráfico
plt.figure(figsize=(10, 6))

plt.plot(df_elem['PEEQ_pct'], df_elem['Volumetric_Strain_pct'], 
         marker='o', linestyle='-', color='purple', markersize=4, 
         label=f'Trajetória - Elemento {elemento_critico}')

# Marcar início e fim da simulação de creep
plt.plot(df_elem['PEEQ_pct'].iloc[0], df_elem['Volumetric_Strain_pct'].iloc[0], 
         marker='s', color='green', markersize=8, label='Início do Creep')
plt.plot(df_elem['PEEQ_pct'].iloc[-1], df_elem['Volumetric_Strain_pct'].iloc[-1], 
         marker='X', color='red', markersize=8, label='Fim da Simulação')

# 5. Linha Crítica de Inversão Volumétrica (Dilatância)
# A linha Y=0 separa a zona de compactação segura da zona de expansão (dano)
plt.axhline(0, color='black', linestyle='--', linewidth=1.5, label='Fronteira Expansão/Compactação')

# 6. Formatação Visual
plt.title('Assinatura de Dilatância: Volumétrica vs. Distorção', fontsize=14, fontweight='bold')
plt.xlabel('Deformação Desviadora / PEEQ (%)', fontsize=12)
plt.ylabel('Deformação Volumétrica (%)', fontsize=12)

# Inverter o eixo Y é uma prática comum em geomecânica para deixar a compactação (negativo) para cima
# Descomente a linha abaixo se preferir essa visualização:
# plt.gca().invert_yaxis() 

plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='best', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  # <-- FALTAVA ESTA IMPORTAÇÃO

# 1. Carregar os dados exportados
df_strain = pd.read_csv("dados_strain_sal.csv")

# 2. Deformação Volumétrica (%)
df_strain['Volumetric_Strain_pct'] = (df_strain['LE11'] + df_strain['LE22'] + df_strain['LE33']) * 100

# 3. Calcular a Deformação Desviadora Total a partir de LE
# Primeiro, acha-se a deformação média (hidrostática)
df_strain['LE_mean'] = (df_strain['LE11'] + df_strain['LE22'] + df_strain['LE33']) / 3.0

# Depois, aplica-se a equação do invariante desviador (fórmula clássica)
df_strain['LE_q'] = np.sqrt((2.0/3.0) * (
    (df_strain['LE11'] - df_strain['LE_mean'])**2 + 
    (df_strain['LE22'] - df_strain['LE_mean'])**2 + 
    (df_strain['LE33'] - df_strain['LE_mean'])**2
))

# Converte para % para o eixo X
df_strain['Deviatoric_pct'] = df_strain['LE_q'] * 100

# --- FALTAVA ESTE BLOCO DE FILTRAGEM ---
# Isolar o elemento crítico
elemento_critico = df_strain['Element_Label'].unique()[0]
df_elem = df_strain[df_strain['Element_Label'] == elemento_critico].sort_values(by='Time_s')
# ---------------------------------------

# 4. Configuração do Gráfico
plt.figure(figsize=(10, 6))

plt.plot(df_elem['Deviatoric_pct'], df_elem['Volumetric_Strain_pct'], 
         marker='o', linestyle='-', color='purple', markersize=4, 
         label=f'Trajetória - Elemento {elemento_critico}')

# Marcar início e fim da simulação de creep
plt.plot(df_elem['Deviatoric_pct'].iloc[0], df_elem['Volumetric_Strain_pct'].iloc[0], 
         marker='s', color='green', markersize=8, label='Início do Creep')
plt.plot(df_elem['Deviatoric_pct'].iloc[-1], df_elem['Volumetric_Strain_pct'].iloc[-1], 
         marker='X', color='red', markersize=8, label='Fim da Simulação')

# 5. Linha Crítica de Inversão Volumétrica (Dilatância)
# A linha Y=0 separa a zona de compactação segura da zona de expansão (dano)
plt.axhline(0, color='black', linestyle='--', linewidth=1.5, label='Fronteira Expansão/Compactação')

# 6. Formatação Visual
plt.title('Assinatura de Dilatância: Volumétrica vs. Distorção', fontsize=14, fontweight='bold')
plt.xlabel('Deformação Desviadora Total (%)', fontsize=12) # Atualizei o rótulo do eixo X
plt.ylabel('Deformação Volumétrica (%)', fontsize=12)

# Inverter o eixo Y é uma prática comum em geomecânica para deixar a compactação (negativo) para cima
# Descomente a linha abaixo se preferir essa visualização:
# plt.gca().invert_yaxis() 

plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='best', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 

# 1. Carregar os dados exportados
df_strain = pd.read_csv("dados_strain_sal.csv")

# 2. Deformação Volumétrica (mStrain)
# Multiplicando por 1000
df_strain['Volumetric_Strain_mStrain'] = (df_strain['LE11'] + df_strain['LE22'] + df_strain['LE33']) * 1000

# 3. Calcular a Deformação Desviadora Total a partir de LE
df_strain['LE_mean'] = (df_strain['LE11'] + df_strain['LE22'] + df_strain['LE33']) / 3.0

df_strain['LE_q'] = np.sqrt((2.0/3.0) * (
    (df_strain['LE11'] - df_strain['LE_mean'])**2 + 
    (df_strain['LE22'] - df_strain['LE_mean'])**2 + 
    (df_strain['LE33'] - df_strain['LE_mean'])**2
))

# Converte para mStrain (x1000)
df_strain['Deviatoric_mStrain'] = df_strain['LE_q'] * 1000

# Isolar o elemento crítico
elemento_critico = df_strain['Element_Label'].unique()[0]
df_elem = df_strain[df_strain['Element_Label'] == elemento_critico].sort_values(by='Time_s')

# 4. Configuração do Gráfico
plt.figure(figsize=(10, 6))

plt.plot(df_elem['Deviatoric_mStrain'], df_elem['Volumetric_Strain_mStrain'], 
         marker='o', linestyle='-', color='purple', markersize=4, 
         label=f'Trajetória - Elemento {elemento_critico}')

# Marcar início e fim da simulação de creep
plt.plot(df_elem['Deviatoric_mStrain'].iloc[0], df_elem['Volumetric_Strain_mStrain'].iloc[0], 
         marker='s', color='green', markersize=8, label='Início do Creep')
plt.plot(df_elem['Deviatoric_mStrain'].iloc[-1], df_elem['Volumetric_Strain_mStrain'].iloc[-1], 
         marker='X', color='red', markersize=8, label='Fim da Simulação')

# 5. Linha Crítica de Inversão Volumétrica (Dilatância)
plt.axhline(0, color='black', linestyle='--', linewidth=1.5, label='Fronteira Expansão/Compactação')

# 6. Formatação Visual
plt.title('Assinatura de Dilatância: Volumétrica vs. Distorção', fontsize=14, fontweight='bold')
plt.xlabel('Deformação Desviadora Total (mStrain)', fontsize=12)
plt.ylabel('Deformação Volumétrica (mStrain)', fontsize=12)

plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='best', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 

# 1. Carregar os dados exportados
df_strain = pd.read_csv("dados_strain_all_sal.csv")

# 2. Deformação Volumétrica (mStrain)
# Multiplicando por 1000
df_strain['Volumetric_Strain_mStrain'] = -(df_strain['LE11'] + df_strain['LE22'] + df_strain['LE33']) * 1000

# 3. Calcular a Deformação Desviadora Total a partir de LE
df_strain['LE_mean'] = (df_strain['LE11'] + df_strain['LE22'] + df_strain['LE33']) / 3.0

df_strain['LE_q'] = np.sqrt((2.0/3.0) * (
    (df_strain['LE11'] - df_strain['LE_mean'])**2 + 
    (df_strain['LE22'] - df_strain['LE_mean'])**2 + 
    (df_strain['LE33'] - df_strain['LE_mean'])**2
))

# Converte para mStrain (x1000)
df_strain['Deviatoric_mStrain'] = df_strain['LE_q'] * 1000

# Isolar o elemento crítico
# elemento_critico = df_strain['Element_Label'].unique()[0]
# elemento_critico = 62
# elemento_critico = 93
# elemento_critico = 124
elemento_critico = 155
df_elem = df_strain[df_strain['Element_Label'] == elemento_critico].sort_values(by='Time_s')

# 4. Configuração do Gráfico (Eixos Invertidos)
plt.figure(figsize=(10, 6))

# Agora Volumétrica no X e Desviadora no Y
plt.plot(df_elem['Volumetric_Strain_mStrain'], df_elem['Deviatoric_mStrain'], 
         marker='o', linestyle='-', color='purple', markersize=4, 
         label=f'Trajetória - Elemento {elemento_critico}')

# Marcar início e fim da simulação de creep (também invertidos)
plt.plot(df_elem['Volumetric_Strain_mStrain'].iloc[0], df_elem['Deviatoric_mStrain'].iloc[0], 
         marker='s', color='green', markersize=8, label='Início do Creep')
plt.plot(df_elem['Volumetric_Strain_mStrain'].iloc[-1], df_elem['Deviatoric_mStrain'].iloc[-1], 
         marker='X', color='red', markersize=8, label='Fim da Simulação')

# 5. Linha Crítica de Inversão Volumétrica (Dilatância)
# Como a Volumétrica agora é o eixo X, a linha zero passa a ser vertical (axvline)
plt.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Fronteira Expansão/Compactação')

# 6. Formatação Visual
plt.title('Assinatura de Dilatância: Distorção vs. Volumétrica', fontsize=14, fontweight='bold')
plt.xlabel('Deformação Volumétrica (mStrain)', fontsize=12)  # X atualizado
plt.ylabel('Deformação Desviadora Total (mStrain)', fontsize=12)  # Y atualizado

plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='best', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 

# 1. Carregar os dados exportados
df_strain = pd.read_csv("dados_strain_all_sal.csv")

# 2. Deformação Volumétrica (mStrain)
# Multiplicando por 1000
df_strain['Volumetric_Strain_mStrain'] = -(df_strain['LE11'] + df_strain['LE22'] + df_strain['LE33']) * 1000

# 3. Calcular a Deformação Desviadora Total a partir de LE
df_strain['LE_mean'] = (df_strain['LE11'] + df_strain['LE22'] + df_strain['LE33']) / 3.0

df_strain['LE_q'] = np.sqrt((2.0/3.0) * (
    (df_strain['LE11'] - df_strain['LE_mean'])**2 + 
    (df_strain['LE22'] - df_strain['LE_mean'])**2 + 
    (df_strain['LE33'] - df_strain['LE_mean'])**2
))

# Converte para mStrain (x1000)
df_strain['Deviatoric_mStrain'] = df_strain['LE_q'] * 1000

# Isolar o elemento crítico (usamos .copy() para o Pandas permitir criar novas colunas sem avisos)
# elemento_critico = df_strain['Element_Label'].unique()[0]
# elemento_critico = 62
# elemento_critico = 93
elemento_critico = 124
# elemento_critico = 155
df_elem = df_strain[df_strain['Element_Label'] == elemento_critico].sort_values(by='Time_s').copy()

# --- NOVO: APLICAÇÃO DA MÉDIA MÓVEL (SUAVIZAÇÃO) ---
# O parâmetro 'window' define a força da suavização (ex: média a cada 10 pontos). 
# Aumente para 20 ou 30 se ainda achar muito serrilhado, ou diminua para 5 se suavizar demais.
tamanho_janela = 40 

df_elem['Volumetric_Smooth'] = df_elem['Volumetric_Strain_mStrain'].rolling(window=tamanho_janela, min_periods=1).mean()
df_elem['Deviatoric_Smooth'] = df_elem['Deviatoric_mStrain'].rolling(window=tamanho_janela, min_periods=1).mean()
# ----------------------------------------------------

# 4. Configuração do Gráfico
plt.figure(figsize=(10, 6))

# Alteramos o plot para usar as novas colunas suavizadas ('_Smooth')
plt.plot(df_elem['Volumetric_Smooth'], df_elem['Deviatoric_Smooth'], 
         marker='.', linestyle='-', color='purple', markersize=3, 
         label=f'Smoothed Trajectory - Element {elemento_critico}')

# Marcar início e fim da simulação de creep (também usando as variáveis suavizadas)
plt.plot(df_elem['Volumetric_Smooth'].iloc[0], df_elem['Deviatoric_Smooth'].iloc[0], 
         marker='s', color='green', markersize=8, label='Beggining of Creep')
plt.plot(df_elem['Volumetric_Smooth'].iloc[-1], df_elem['Deviatoric_Smooth'].iloc[-1], 
         marker='X', color='red', markersize=8, label='End of Simulation')

# 5. Linha Crítica de Inversão Volumétrica (Dilatância)
plt.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Expansion/Compaction Boundary')

# 6. Formatação Visual
plt.title('Dilatancy Signature: Distortion vs. Volumetric', fontsize=14, fontweight='bold')
plt.xlabel('Volumetric Strain (mStrain)', fontsize=12) 
plt.ylabel('Total Deviatoric Strain (mStrain)', fontsize=12) 

plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='best', frameon=True, shadow=True)

plt.tight_layout()
plt.show()